# 3.1 — Tokenization avec spaCy et NLTK

## Objectifs pédagogiques

À la fin de ce notebook, vous saurez :
- expliquer la tokenisation et son rôle en NLP ;
- utiliser `spaCy` pour découper un texte en tokens ;
- examiner les propriétés d’un token (`text`, `shape_`, `is_alpha`, `is_stop`) ;
- comprendre les différences entre une chaîne Python, `Doc`, `Token` et `Span` ;
- comparer la tokenisation de `spaCy`, `NLTK` et `str.split()` ;
- tester des exemples en anglais et en arabe.

> **Important :** les résultats exacts peuvent varier selon la version de la bibliothèque et le modèle linguistique installé.


## 1. Installer les bibliothèques

Dans Jupyter ou Google Colab, exécutez cette cellule si les bibliothèques ne sont pas déjà installées.

In [ ]:
%pip install -q spacy nltk

## 2. Charger spaCy et son modèle anglais

Le paquet `spacy` fournit le moteur de traitement, tandis que `en_core_web_sm` est un modèle linguistique anglais. Il faut télécharger ce modèle une première fois.

Si le modèle est déjà installé, vous pouvez ignorer la cellule de téléchargement.

In [ ]:
import sys

# Installer le petit modèle anglais de spaCy
!{sys.executable} -m spacy download en_core_web_sm

In [ ]:
import spacy

# Charger le modèle anglais
nlp = spacy.load("en_core_web_sm")

print("Version de spaCy :", spacy.__version__)
print("Pipeline :", nlp.pipe_names)

## 3. Qu’est-ce que la tokenisation ?

La **tokenisation** consiste à découper un texte en unités appelées *tokens*. Un token peut être un mot, un signe de ponctuation, un nombre ou une autre unité reconnue par le tokenizer.

Avec spaCy, l’objet `nlp` traite une chaîne de caractères et retourne un objet `Doc`, qui contient les tokens et, selon le pipeline, des annotations linguistiques.

In [ ]:
text = "Tesla is looking at buying a U.S. startup for $69 million."

# Traiter le texte avec le pipeline spaCy
doc = nlp(text)

# Afficher le type de l'objet retourné
print(type(doc))

# Afficher chaque token
for token in doc:
    print(token.text)

## 4. Examiner les propriétés des tokens

Chaque objet `Token` expose plusieurs propriétés utiles :
- `token.text` : forme exacte du token ;
- `token.shape_` : représentation simplifiée de sa forme ;
- `token.is_alpha` : indique si le token est composé uniquement de caractères alphabétiques ;
- `token.is_stop` : indique si le token appartient à la liste de mots vides (*stopwords*) du modèle.

Un mot vide peut être important dans le sens d’une phrase ; il ne faut donc pas le supprimer automatiquement dans tous les projets.

In [ ]:
for token in doc:
    print(f"Token : {token.text!r}")
    print(f"  Forme (shape_) : {token.shape_}")
    print(f"  Alphabétique ? : {token.is_alpha}")
    print(f"  Stopword ?     : {token.is_stop}")
    print("-" * 35)

## 5. Accéder à un token par son indice

Les tokens d’un `Doc` sont indexés à partir de `0`, comme les éléments d’une liste Python. `doc[0]` désigne le premier token.

In [ ]:
# Afficher les premiers tokens avec leurs indices
for i, token in enumerate(doc[:8]):
    print(i, "->", token.text)

## 6. Sélectionner un groupe de tokens avec `Span`

Une tranche telle que `doc[2:5]` retourne un objet `Span`, c’est-à-dire une séquence contiguë de tokens. L’indice de fin est exclu : `doc[2:5]` contient les tokens d’indices 2, 3 et 4.

In [ ]:
# Sélectionner les tokens d'indices 2, 3 et 4
span = doc[2:5]

print("Texte du span :", span.text)
print("Type :", type(span))

## 7. Tokenisation de la ponctuation, des contractions et des URL

Le tokenizer de spaCy applique des règles linguistiques. Il peut, par exemple, séparer certains signes de ponctuation tout en conservant des unités comme les adresses e-mail selon ses règles.

Observez le résultat : il est préférable de vérifier les tokens plutôt que de supposer comment une chaîne sera découpée.

In [ ]:
examples = [
    '"We\'re moving to L.A.!"',
    "We're here to help! Send snail-mail, email support@oursite.com or visit http://www.oursite.com!",
    "A 5km NYC cab ride costs $10.30",
    "Let's visit St. Louis in the U.S. next year."
]

for example in examples:
    print("Texte :", example)
    print("Tokens:", [token.text for token in nlp(example)])
    print("-" * 60)

## 8. La tokenisation ne modifie pas automatiquement le texte original

Le texte source reste une chaîne de caractères. Le `Doc` est une représentation tokenisée du texte.

**Attention :** les tokens d’un objet `Doc` ne peuvent pas être remplacés par une affectation telle que `doc[3] = ...`. Si vous souhaitez modifier le texte, modifiez la chaîne avant de la traiter à nouveau.

In [ ]:
text1 = "My dinner was horrible."
text2 = "Your dinner was delicious."

doc1 = nlp(text1)
doc2 = nlp(text2)

print("Texte 1 :", [token.text for token in doc1])
print("Texte 2 :", [token.text for token in doc2])

# Pour modifier le texte, on modifie la chaîne puis on relance le pipeline
revised_text = text1.replace("horrible", "delicious")
revised_doc = nlp(revised_text)

print("Texte modifié :", revised_text)
print("Nouveaux tokens:", [token.text for token in revised_doc])

## 9. Tokenisation avec NLTK

NLTK propose également des tokenizers. Selon la version, `word_tokenize()` peut nécessiter les ressources `punkt` et `punkt_tab`.

La sortie de NLTK peut être différente de celle de spaCy : les deux bibliothèques utilisent des règles différentes.

In [ ]:
import nltk

# Télécharger les ressources de tokenisation requises par les versions récentes
nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)

from nltk.tokenize import word_tokenize

example_text = """
Hello Mr. Smith, how are you doing today? The weather is great,
and Python is awesome. The sky is pinkish-blue. You shouldn't eat cardboard.
"""

tokens_nltk = word_tokenize(example_text, language="english")
print(tokens_nltk)

## 10. Comparer `str.split()`, NLTK et spaCy

`str.split()` sépare principalement selon les espaces. Il ne traite pas spécialement la ponctuation. Les bibliothèques NLP appliquent des règles de tokenisation plus élaborées.

Il n’existe pas un découpage universellement correct pour toutes les tâches : choisissez la méthode en fonction de l’application.

In [ ]:
text = "Hello Mr. Smith! You shouldn't pay $10.30."

# Méthode simple : séparation sur les espaces
print("str.split():", text.split())

# Tokenisation NLTK
print("NLTK       :", word_tokenize(text, language="english"))

# Tokenisation spaCy
print("spaCy      :", [token.text for token in nlp(text)])

## 11. Tokenisation d’un texte plus long

Pour un texte long, la tokenisation permet de parcourir le document, de compter les tokens ou de comparer des segments. Les exemples suivants utilisent un extrait littéraire à titre de démonstration.

In [ ]:
example_text = """
Thomas Gradgrind, sir. A man of realities. A man of facts and calculations.
A man who proceeds upon the principle that two and two are four.
"""

doc_long = nlp(example_text)

print("Nombre de tokens :", len(doc_long))
print("Premiers 20 tokens :", [token.text for token in doc_long[:20]])

# Découper le texte en phrases à l'aide du pipeline spaCy
for sentence in doc_long.sents:
    print("-", sentence.text)

## 12. Tokenisation de textes en arabe

Le modèle `en_core_web_sm` est un modèle **anglais**. Il peut produire des tokens à partir de caractères arabes, mais ses annotations linguistiques ne sont pas adaptées à l’arabe.

Pour illustrer uniquement la tokenisation, on peut créer un pipeline spaCy vide avec le tokenizer par défaut. Pour une analyse linguistique arabe, il faut choisir un modèle ou un outil adapté à cette langue.

In [ ]:
arabic_text = "يعد الذكاء الاصطناعي من العلوم التي يتسارع التطور فيها منذ عام 2005."

# Pipeline spaCy vide : utile ici pour illustrer la tokenisation sans modèle anglais
nlp_blank_ar = spacy.blank("ar")
doc_ar = nlp_blank_ar(arabic_text)

for token in doc_ar:
    print(
        f"Token: {token.text!r} | "
        f"Shape: {token.shape_} | "
        f"Alphabetic: {token.is_alpha}"
    )

## 13. Tokeniser un texte arabe avec NLTK

NLTK peut également découper un texte arabe. Le résultat dépend du tokenizer utilisé ; cette opération ne constitue pas une analyse morphologique ou syntaxique de l’arabe.

In [ ]:
arabic_text = """
أبو عبد الله محمد بن موسى الخوارزمي عالم رياضيات وفلك وجغرافيا مسلم.
يعتبر من أوائل علماء الرياضيات المسلمين.
"""

# Pour cet exemple arabe, utiliser un tokenizer simple basé sur les espaces.
# Il ne réalise pas une analyse morphologique et conserve la ponctuation attachée aux mots.
from nltk.tokenize import WhitespaceTokenizer

tokens_ar = WhitespaceTokenizer().tokenize(arabic_text)
print(tokens_ar)

## 14. Résumé des objets et propriétés

| Élément | Signification |
|---|---|
| `nlp(text)` | Applique le pipeline spaCy et retourne un `Doc` |
| `Doc` | Représentation tokenisée du texte |
| `Token` | Unité individuelle du document |
| `Span` | Séquence contiguë de tokens |
| `token.text` | Texte exact du token |
| `token.shape_` | Forme simplifiée du token |
| `token.is_alpha` | Vrai si le token est alphabétique |
| `token.is_stop` | Vrai si le token est un mot vide du modèle |
| `word_tokenize(text)` | Tokenisation avec NLTK |
| `text.split()` | Découpage simple sur les espaces |

## 15. Exercices

1. Tokenisez la phrase `Natural Language Processing is useful!` avec spaCy et NLTK.
2. Affichez le texte, la forme (`shape_`) et la propriété `is_alpha` de chaque token.
3. Extrayez le groupe de tokens correspondant à `New York` dans `I live in New York.`.
4. Comparez le résultat de `split()` et de spaCy pour `We're learning NLP, version 2.0!`.
5. Tokenisez un paragraphe arabe avec `spacy.blank("ar")` puis avec NLTK. Comparez les sorties.
6. Expliquez pourquoi on ne peut pas écrire `doc[3] = another_token`.

### Exemple de solution pour l'exercice 3


In [ ]:
text = "I live in New York."
doc = nlp(text)

# Afficher les tokens avec leurs indices pour identifier la tranche
for i, token in enumerate(doc):
    print(i, token.text)

# Adapter les indices en fonction du résultat affiché
# Ici, on sélectionne le groupe correspondant à "New York"
print(doc[3:5].text)

## À retenir

- La tokenisation est une étape fondamentale du prétraitement NLP.
- `spaCy`, `NLTK` et `str.split()` ne découpent pas nécessairement le texte de la même façon.
- Un `Doc` spaCy contient des tokens ; une tranche retourne un `Span`.
- Le modèle de langue doit être adapté au texte analysé.
- La tokenisation seule ne donne pas automatiquement le sens des mots.
